# Sentiment Classifier Bias Audit Across Demographic Subgroups
**Text Analytics Term Project**

This notebook builds a reproducible bias-auditing workflow for a binary sentiment classifier. It creates a small **synthetic demonstration dataset** with text, sentiment labels, and illustrative demographic attributes, trains a TF–IDF + Logistic Regression classifier, and compares performance across subgroups.

> **Important:** The included data are synthetic and are only for demonstrating the workflow. Results are not evidence about any real population. For a real audit, replace the data-generation cell with an appropriately licensed, consented, representative dataset and document the limitations of demographic labels.

## Objectives
1. Prepare text and labels for sentiment classification.
2. Train a baseline model using a stratified train/test split.
3. Evaluate overall performance and subgroup performance.
4. Compare error rates and flag disparities using configurable thresholds.
5. Export tables and plots for the project report.

**Tools:** Python, pandas, scikit-learn, matplotlib, Jupyter Notebook.

## 1. Imports and configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from IPython.display import display

RANDOM_STATE = 42
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
pd.set_option("display.max_colwidth", 100)
print("Environment ready. Random state:", RANDOM_STATE)

## 2. Create a synthetic dataset

In [ ]:
# Synthetic, intentionally illustrative data; not collected from real people.
# Group-associated wording is included to demonstrate why subgroup auditing matters.
rng = np.random.default_rng(RANDOM_STATE)

positive_templates = [
    "I absolutely love this product", "This is wonderful and excellent",
    "Really happy with the service", "Great quality, I recommend it",
    "Fantastic experience, very good", "I am pleased with this purchase",
    "Brilliant result and lovely design", "This works well and feels amazing"
]
negative_templates = [
    "I really hate this product", "This is awful and disappointing",
    "Very unhappy with the service", "Poor quality, do not recommend",
    "Terrible experience, very bad", "I regret this purchase",
    "Horrible result and frustrating design", "This does not work well"
]
neutral_templates = [
    "The product arrived yesterday", "I used the service today",
    "The package contains two items", "This is the current version",
    "The order was processed in the morning", "The item has a standard design"
]
groups_gender = ["Woman", "Man", "Non-binary"]
groups_dialect = ["Standard-style", "Colloquial-style"]

rows = []
for i in range(900):
    label = int(rng.random() < 0.5)
    if rng.random() < 0.12:
        text = rng.choice(neutral_templates)
        # For this binary demo, neutral statements are assigned a weak/ambiguous label.
        label = int(rng.random() < 0.5)
    else:
        text = rng.choice(positive_templates if label else negative_templates)

    gender = rng.choice(groups_gender, p=[0.46, 0.46, 0.08])
    dialect = rng.choice(groups_dialect, p=[0.65, 0.35])

    # Illustrative style variation; do not interpret as a real demographic language pattern.
    if dialect == "Colloquial-style":
        if label:
            text = text.replace("wonderful", "so good").replace("excellent", "awesome").replace("really", "so")
            text = text + rng.choice(["!!", " :)", " lol", ""])
        else:
            text = text.replace("awful", "so bad").replace("terrible", "really bad")
            text = text + rng.choice(["!!", " :(", " smh", ""])
    if gender == "Non-binary" and rng.random() < 0.25:
        text = "Honestly, " + text.lower()

    rows.append({"text": text, "sentiment": label, "gender_group": gender, "style_group": dialect})

df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
df["sentiment_label"] = df["sentiment"].map({0: "Negative", 1: "Positive"})
print("Dataset shape:", df.shape)
display(df.head(10))
display(df.groupby(["gender_group", "sentiment_label"]).size().unstack(fill_value=0))

## 3. Dataset checks

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))
print("\nClass distribution:")
display(df["sentiment_label"].value_counts(normalize=True).rename("proportion").to_frame())
print("\nSubgroup sizes:")
display(df["gender_group"].value_counts().to_frame("rows"))
display(df["style_group"].value_counts().to_frame("rows"))

## 4. Train/test split and model

In [ ]:
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    df["text"], df["sentiment"], df[["gender_group", "style_group"]],
    test_size=0.25, random_state=RANDOM_STATE, stratify=df["sentiment"]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=True)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight=None, random_state=RANDOM_STATE))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"], zero_division=0))

## 5. Overall metrics

In [ ]:
def binary_metrics(y_true, y_hat):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    return {
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_hat),
        "precision_positive": precision_score(y_true, y_hat, zero_division=0),
        "recall_positive": recall_score(y_true, y_hat, zero_division=0),
        "f1_positive": f1_score(y_true, y_hat, zero_division=0),
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else np.nan,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) else np.nan,
        "positive_prediction_rate": float(np.mean(y_hat))
    }

overall = pd.DataFrame([binary_metrics(y_test, y_pred)], index=["Overall"])
display(overall.round(3))
overall.to_csv(os.path.join(OUTPUT_DIR, "overall_metrics.csv"))

## 6. Subgroup performance audit

In [ ]:
test_results = meta_test.copy().reset_index(drop=True)
test_results["y_true"] = np.asarray(y_test)
test_results["y_pred"] = np.asarray(y_pred)
test_results["text"] = np.asarray(X_test)

def subgroup_audit(results, group_col, min_group_n=10):
    records = []
    for group, part in results.groupby(group_col):
        metrics = binary_metrics(part["y_true"], part["y_pred"])
        metrics[group_col] = group
        metrics["small_sample_warning"] = metrics["n"] < min_group_n
        records.append(metrics)
    return pd.DataFrame(records).set_index(group_col).sort_index()

gender_audit = subgroup_audit(test_results, "gender_group")
style_audit = subgroup_audit(test_results, "style_group")
print("Audit by gender_group:")
display(gender_audit.round(3))
print("Audit by style_group:")
display(style_audit.round(3))

gender_audit.to_csv(os.path.join(OUTPUT_DIR, "gender_subgroup_metrics.csv"))
style_audit.to_csv(os.path.join(OUTPUT_DIR, "style_subgroup_metrics.csv"))

## 7. Disparity summary and configurable flags

In [ ]:
# These are screening thresholds, not legal or universal definitions of fairness.
ACCURACY_GAP_THRESHOLD = 0.10
ERROR_RATE_GAP_THRESHOLD = 0.10

def disparity_summary(audit, group_col):
    metrics = ["accuracy", "false_positive_rate", "false_negative_rate", "positive_prediction_rate"]
    rows = []
    for metric in metrics:
        values = audit[metric].dropna()
        gap = float(values.max() - values.min()) if len(values) >= 2 else np.nan
        threshold = ACCURACY_GAP_THRESHOLD if metric == "accuracy" else ERROR_RATE_GAP_THRESHOLD
        rows.append({
            "grouping": group_col, "metric": metric,
            "min_value": values.min() if len(values) else np.nan,
            "max_value": values.max() if len(values) else np.nan,
            "absolute_gap": gap, "screening_threshold": threshold,
            "flag_for_review": bool(gap > threshold) if not np.isnan(gap) else False
        })
    return pd.DataFrame(rows)

disparities = pd.concat([
    disparity_summary(gender_audit, "gender_group"),
    disparity_summary(style_audit, "style_group")
], ignore_index=True)
display(disparities.round(3))
disparities.to_csv(os.path.join(OUTPUT_DIR, "disparity_summary.csv"), index=False)
print("A flag means investigate further; it does not prove discrimination.")

## 8. Visualize subgroup metrics

In [ ]:
def plot_metric(audit, metric, title, filename):
    ax = audit[metric].plot(kind="bar", ylim=(0, 1), figsize=(8, 4), rot=20)
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()

plot_metric(gender_audit, "accuracy", "Accuracy by gender subgroup (synthetic demo)", "accuracy_by_gender.png")
plot_metric(gender_audit, "false_positive_rate", "False-positive rate by gender subgroup", "fpr_by_gender.png")
plot_metric(style_audit, "accuracy", "Accuracy by writing-style subgroup (synthetic demo)", "accuracy_by_style.png")
plot_metric(style_audit, "false_negative_rate", "False-negative rate by writing-style subgroup", "fnr_by_style.png")

## 9. Error inspection

In [ ]:
errors = test_results[test_results["y_true"] != test_results["y_pred"]].copy()
errors["true_label"] = errors["y_true"].map({0: "Negative", 1: "Positive"})
errors["predicted_label"] = errors["y_pred"].map({0: "Negative", 1: "Positive"})
print(f"Misclassified test examples: {len(errors)} / {len(test_results)}")
display(errors[["text", "gender_group", "style_group", "true_label", "predicted_label"]].head(20))
errors.to_csv(os.path.join(OUTPUT_DIR, "misclassified_examples.csv"), index=False)

## 10. Interpretation and limitations

### How to interpret the audit
- Compare **accuracy** to see whether the model is correct at different rates.
- Compare **false-positive rate (FPR)**: among truly negative examples, how often is a positive sentiment predicted?
- Compare **false-negative rate (FNR)**: among truly positive examples, how often is a negative sentiment predicted?
- Compare **positive prediction rate** as a descriptive measure, not a standalone fairness verdict.
- Treat small subgroup sizes as uncertain; report denominators and consider confidence intervals before drawing conclusions.

### Limitations
1. The data are synthetic, small, and generated from templates.
2. The demographic fields are illustrative categories and must not be inferred from names, text, or appearance in a real deployment.
3. The binary sentiment setup simplifies nuanced or neutral sentiment.
4. A metric gap may reflect sampling noise, label quality, text domain, or model behavior; it does not alone establish unfairness.
5. Fairness metrics can conflict. Thresholds should be chosen with domain experts and affected stakeholders.

### Suggested next steps for a real audit
Use a representative, consented dataset; preregister subgroup definitions; check subgroup sample sizes; add bootstrap confidence intervals; evaluate multiple models and thresholds; document data provenance; and conduct human review before deployment.

## 11. Export audit artifacts

In [ ]:
# Save a compact final summary for the report/demo.
summary = {
    "dataset_rows": int(len(df)),
    "test_rows": int(len(y_test)),
    "overall_accuracy": float(accuracy_score(y_test, y_pred)),
    "overall_f1_positive": float(f1_score(y_test, y_pred, zero_division=0)),
    "gender_flags": int(disparities.query("grouping == 'gender_group'")["flag_for_review"].sum()),
    "style_flags": int(disparities.query("grouping == 'style_group'")["flag_for_review"].sum())
}
summary_df = pd.DataFrame([summary])
display(summary_df.round(3))
summary_df.to_csv(os.path.join(OUTPUT_DIR, "project_summary.csv"), index=False)
print("Saved CSV tables and PNG plots in:", OUTPUT_DIR)

## Conclusion
This project demonstrates an end-to-end **bias-auditing tool** for sentiment classification. It reports aggregate and subgroup metrics, surfaces configurable disparity flags, and saves tables/plots for reproducible review. The synthetic example is a teaching scaffold; responsible conclusions require real, suitable data and uncertainty analysis.